# Notebook 066: Medallion Architecture: Gold Business Aggregations and Dimensional Serving

**Module:** Module 07: Lakehouse Data Modeling, CDC & SCD Implementation  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 19: Databricks Dashboard from Scratch & Video 24: Metric Views  

---

## 1. Business Scenario & Real-World Context
Executive leadership and Power BI dashboards require pre-aggregated quarterly vendor performance scorecards: total invoiced volume, approval rate percentages, average turnaround time, and vendor tier classifications.

## 2. Technical Requirement & Problem Statement
Build the Gold layer of the Medallion Architecture: read from conformed Silver Delta tables, compute executive business metrics, apply business KPIs and risk score calculations, and write high-performance Gold aggregate tables optimized for BI consumption.

## 3. Spark & Databricks Architecture Notes
The Gold layer consists of project-specific, business-level aggregated tables, star schema dimension and fact tables, or feature store tables. Gold tables are typically optimized with Z-Ordering or Liquid Clustering to ensure sub-second dashboard query response times.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_066_Medallion_Architecture_Gold_Business_Aggregations_and_Dimensional_Serving")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Reading from Silver Layer Invoices
Create realistic Silver invoice dataset.

In [ ]:
silver_data = [
    ("INV-1", "V-01", 1500.0, "APPROVED", "2023-10-01"),
    ("INV-2", "V-01", 3200.0, "APPROVED", "2023-10-15"),
    ("INV-3", "V-01", 800.0,  "REJECTED", "2023-10-20"),
    ("INV-4", "V-02", 450.0,  "APPROVED", "2023-10-05"),
    ("INV-5", "V-02", 900.0,  "PENDING",  "2023-10-25"),
    ("INV-6", "V-03", 12500.0,"APPROVED", "2023-10-12")
]

schema = "invoice_no STRING, vendor_id STRING, amount DOUBLE, status STRING, invoice_date STRING"
df_silver_in = spark.createDataFrame(silver_data, schema)
df_silver_in.show()

## 6. Computing Gold KPI Aggregations and Business Tiers
Calculate approval rate, total volume, and classify vendor risk tiers.

In [ ]:
df_gold_kpis = (
    df_silver_in
    .groupBy("vendor_id")
    .agg(
        F.count("invoice_no").alias("total_invoices_submitted"),
        F.round(F.sum("amount"), 2).alias("total_invoiced_spend"),
        F.round(F.avg("amount"), 2).alias("avg_invoice_value"),
        F.sum(F.when(F.col("status") == "APPROVED", 1).otherwise(0)).alias("approved_invoices_count"),
        F.sum(F.when(F.col("status") == "REJECTED", 1).otherwise(0)).alias("rejected_invoices_count")
    )
    .withColumn(
        "approval_rate_pct",
        F.round((F.col("approved_invoices_count") / F.col("total_invoices_submitted")) * 100, 1)
    )
    .withColumn(
        "vendor_volume_tier",
        F.when(F.col("total_invoiced_spend") >= 10000, "STRATEGIC_TIER_1")
         .when(F.col("total_invoiced_spend") >= 2000, "PREFERRED_TIER_2")
         .otherwise("STANDARD_TIER_3")
    )
)

print("Gold Vendor Performance Scorecard:")
df_gold_kpis.show(truncate=False)

## 7. Writing to Gold Delta Lake Table with Liquid Clustering
Save to Gold destination ready for BI and semantic metrics.

In [ ]:
gold_path = "/tmp/lakehouse_gold_066"
if Path(gold_path).exists():
    shutil.rmtree(gold_path)

df_gold_kpis.write.format("delta").mode("overwrite").save(gold_path)

print("Gold Table successfully landed. Ready for Power BI / Tableau consumption!")

## 9. Key Architectural Takeaways
- **Gold tables represent domain-specific business aggregates and dimensional models designed for end-user analytics.**
- **Pre-aggregating KPIs in Gold reduces compute consumption on BI tools and guarantees single-version-of-truth metrics.**
- **Apply clustering (`CLUSTER BY`) on Gold tables based on typical dashboard filter columns.**

## 10. Senior Data Engineering Interview Questions
1. Why should BI tools query Gold tables rather than directly querying Silver tables?
2. How does the Medallion Architecture ensure compliance with GDPR 'Right to be Forgotten' across Bronze, Silver, and Gold?
3. What is the difference between a Gold table and a materialized view in Databricks?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
